# Tips 20：Googleスプレッドシートを集計

**業務の困りごと**：共同編集するSheetの数値を集計する。

**このTipsでできること**：表の「売上」は文字列なので `to_numeric` で数値に変えます。その後、店舗別に合計してExcelに保存します。

**使用ライブラリ**：gspread, pandas

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

Googleスプレッドシートの行を模したデータです。最初の行が列見出しです。

最後の `head()` で先頭の行を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
import pandas as pd
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 1行目を見出しとしたSheet風のデータ
rows=[['店舗','売上'],['鹿児島','1200'],['霧島','900'],['鹿児島','300']]
print('練習用のSheetデータ（先頭5行）:')
pd.DataFrame(rows[1:],columns=rows[0]).head()


## 2. 自動処理

表の「売上」は文字列なので `to_numeric` で数値に変えます。その後、店舗別に合計してExcelに保存します。

**結果を見るポイント**：店舗別合計の総額は **2,400円** です。


In [ ]:
# 見出し行とデータ行を表に分ける
frame=pd.DataFrame(rows[1:],columns=rows[0])
# 文字列の売上を数値へ変える
frame['売上']=pd.to_numeric(frame['売上'],errors='raise')
# 店舗別に売上を合計する
summary=frame.groupby('店舗',as_index=False)['売上'].sum()
out=base/'Tips20_Sheet集計.xlsx'; summary.to_excel(out,index=False)
print(summary)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

店舗別合計の総額は **2,400円** です。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert summary['売上'].sum()==2400 and out.exists()
print('確認OK')


## 発展：自分のGoogleスプレッドシートから取得（Colab専用・任意）
自分がアクセスできるSheetのURLを入力して、認証した後で実行します。列見出しは「店舗」「売上」を想定します。


In [ ]:
# 自分のSheetを使うときだけコメントを外して実行
# %pip -q install gspread
# from google.colab import auth
# auth.authenticate_user()
# import google.auth, gspread
# credentials,_=google.auth.default()
# gc=gspread.authorize(credentials)
# sheet_url='ここに自分のGoogleスプレッドシートのURL'
# worksheet=gc.open_by_url(sheet_url).sheet1
# live=pd.DataFrame(worksheet.get_all_records())
# live['売上']=pd.to_numeric(live['売上'],errors='raise')
# live.groupby('店舗',as_index=False)['売上'].sum()


## 4. 持ち帰り・練習

**練習問題**：Googleアカウントで認証し、自分のSheetの列名を合わせて読み込んでみましょう。

次の連携セルはColabで本人が権限を与える場合だけ実行します。既定ではアクセスしません。

**実務で使う前に**：自分のSheetに接続する発展セルは認証とURLの設定が必要です。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
